**Import Liabraries**

In [31]:

import requests
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from datetime import datetime, timedelta
import pytz

In [32]:
API_KEY = '2d49187691fbc44199f3ede4fcb67348'
BASE_URL = 'https://api.openweathermap.org/data/2.5/'

**1. FETCH CURRENT WEATHER**

In [33]:
def get_current_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()

    return {
        'city': data.get('name', city),
        'current_temp': round(data['main']['temp']),
        'feels_like': round(data['main']['feels_like']),
        'temp_min': round(data['main']['temp_min']),
        'temp_max': round(data['main']['temp_max']),
        'humidity': round(data['main']['humidity']),
        'description': data['weather'][0]['description'],
        'country': data['sys']['country'],
        'WindGustDir': data['wind'].get('deg', 0),
        'Wind_Gust_Speed': data['wind'].get('speed', 0),
        'Pressure': data['main']['pressure']
    }

**2. READ HISTORICAL DATA**

In [34]:
def read_historical_data(filename):
    df = pd.read_csv(filename)
    df = df.dropna()
    df = df.drop_duplicates()
    return df

**3. PREPARE DATA FOR RAIN MODEL**

In [41]:
def prepare_data(data):
    data['Temp'] = data['temperature']
    data['WindGustSpeed'] = data['wind_speed']
    data['RainTomorrow'] = data['weather_condition'].apply(lambda x: 1 if 'rain' in x.lower() else 0)

    x = data[['Temp', 'WindGustSpeed', 'humidity', 'pressure']]
    y = data['RainTomorrow']

    return x, y

**4. TRAIN RAIN PREDICTION MODEL**

In [19]:
def train_rain_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(x_train, y_train)

    return model

**5. PREPARE REGRESSION DATA**

In [36]:
def prepare_regression_data(data, feature):
    x, y = [], []

    for i in range(len(data) - 1):
        x.append(data[feature].iloc[i])
        y.append(data[feature].iloc[i + 1])

    x = np.array(x).reshape(-1, 1)
    y = np.array(y)

    return x, y

**6. TRAIN REGRESSION MODEL**

In [37]:
def train_regression_model(x, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x, y)
    return model


**7. PREDICT FUTURE VALUES**

In [38]:
def predict_future(model, current_value):
    predictions = [current_value]

    for _ in range(5):
        next_val = model.predict(np.array([[predictions[-1]]]))[0]
        predictions.append(next_val)

    return predictions[1:]

8. MAIN WEATHER FUNCTION

In [42]:
def weather_view():
    city = input("Enter the city name: ")
    current_weather = get_current_weather(city)

    historical_data = read_historical_data('/weather_data.csv')

    # Adjusted call to prepare_data - no more 'le'
    x, y = prepare_data(historical_data)
    rain_model = train_rain_model(x, y)

    # Removed WIND DIRECTION CONVERSION block

    current_data = {
        'Temp': current_weather['current_temp'],
        'WindGustSpeed': current_weather['Wind_Gust_Speed'],
        'humidity': current_weather['humidity'],
        'pressure': current_weather['Pressure']
    }

    current_df = pd.DataFrame([current_data])
    rain_prediction = rain_model.predict(current_df)[0]

    # REGRESSION MODELS - updated feature names
    x_temp, y_temp = prepare_regression_data(historical_data, 'temperature')
    x_hum, y_hum = prepare_regression_data(historical_data, 'humidity')

    temp_model = train_regression_model(x_temp, y_temp)
    hum_model = train_regression_model(x_hum, y_hum)

    future_temp = predict_future(temp_model, current_weather['current_temp'])
    future_humidity = predict_future(hum_model, current_weather['humidity'])

    timezone = pytz.timezone("Asia/Karachi")
    now = datetime.now(timezone)
    next_hour = now.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)

    future_times = [
        (next_hour + timedelta(hours=i)).strftime("%H:00")
        for i in range(5)
    ]

    # OUTPUT
    print(f"\nCity: {current_weather['city']}, {current_weather['country']}")
    print(f"Current Temperature: {current_weather['current_temp']}°C")
    print(f"Feels Like: {current_weather['feels_like']}°C")
    print(f"Humidity: {current_weather['humidity']}%")
    print(f"Weather: {current_weather['description']}")
    print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")

    print("\nFuture Temperature Prediction:")
    for t, temp in zip(future_times, future_temp):
        print(f"{t}: {round(temp, 1)}°C")

    print("\nFuture Humidity Prediction:")
    for t, hum in zip(future_times, future_humidity):
        print(f"{t}: {round(hum, 1)}%")

In [43]:
weather_view()

Enter the city name: surat

City: Surat, IN
Current Temperature: 30°C
Feels Like: 30°C
Humidity: 39%
Weather: haze
Rain Prediction: No

Future Temperature Prediction:
12:00: 30.3°C
13:00: 30.3°C
14:00: 30.3°C
15:00: 30.3°C
16:00: 30.3°C

Future Humidity Prediction:
12:00: 27.0%
13:00: 27.0%
14:00: 27.0%
15:00: 27.0%
16:00: 27.0%
